# Модель 1: TF-IDF + логистическая регрессия

Цели из установочного отчёта: **F1-macro ≥ 0.80** по тональности и **≥ 0.75** по категориям, 1000 отзывов меньше чем за минуту.

Ноутбук работает локально (из папки `ml/notebooks`) и в Google Colab. Для Colab выполните первую ячейку.

In [ ]:
# Только для Colab: скачать проект и поставить библиотеки.
import sys, os
if "google.colab" in sys.modules and not os.path.exists("ai-customer-insight"):
    !git clone https://github.com/3092334al-del/ai-customer-insight.git
    %cd ai-customer-insight/ml/notebooks
    !pip install -q -r ../requirements.txt

In [ ]:
import sys
from pathlib import Path

ML = Path.cwd().parent
sys.path.insert(0, str(ML))

import pandas as pd
from aci_ml import baseline
from aci_ml.dataset import load_dataset

## 1. Данные

Положите файлы SemEval-2016 в `ml/data/raw/` (см. `ml/data/README.md`). Свою разметку (CSV единого формата) добавьте в список `OWN`.

Если корпуса ещё нет, ноутбук возьмёт маленький тестовый пример — цифры на нём ничего не значат, он нужен только чтобы проверить, что всё запускается.

In [ ]:
TRAIN = ML / "data" / "raw" / "train.xml"
TEST = ML / "data" / "raw" / "test.xml"
OWN = []  # например: [ML / "data" / "наша_разметка.csv"]
LEVEL = "sentence"  # или "review" — целый отзыв одной строкой

if TRAIN.exists():
    train_df = pd.concat([load_dataset(TRAIN, LEVEL)] + [load_dataset(p) for p in OWN], ignore_index=True)
    test_df = load_dataset(TEST, LEVEL) if TEST.exists() else None
else:
    print("Корпуса нет — используем тестовый пример")
    train_df = pd.concat([load_dataset(ML / "tests" / "fixtures" / "semeval_ru_sample.xml"),
                          load_dataset(ML / "data" / "разметка_шаблон.csv")], ignore_index=True)
    test_df = None

if test_df is None:
    train_df, test_df = baseline.split(train_df)

print(f"Обучение: {len(train_df)}, проверка: {len(test_df)}")
train_df.head()

Сколько примеров каждого класса. Если какого-то класса очень мало, модель будет на нём ошибаться — это стоит обсудить в отчёте.

In [ ]:
print(train_df["sentiment"].value_counts())
print()
print((train_df[["service", "food", "price", "atmosphere"]] != "absence").sum().rename("упоминаний"))

## 2. Обучение и проверка

In [ ]:
model = baseline.train(train_df)
metrics = baseline.evaluate(model, test_df)
print(baseline.format_report(metrics))

## 3. Где модель ошибается

Разбор ошибок — самая полезная часть для отчёта: по нему видно, что модель не понимает (ирония, отрицание, смешанные отзывы).

In [ ]:
pred = baseline.predict(model, test_df["text"])
errors = test_df.assign(predicted=[p["sentiment"] for p in pred])
errors = errors[errors["predicted"] != errors["sentiment"]][["text", "sentiment", "predicted"]]
pd.set_option("display.max_colwidth", 120)
errors.head(20)

## 4. Свои примеры

In [ ]:
examples = [
    "Официант принёс холодный суп и даже не извинился",
    "Неплохо, но дороговато",
    "Ну конечно, «быстрая» доставка — всего полтора часа",
]
for text, result in zip(examples, baseline.predict(model, examples)):
    print(f"{result['sentiment']:9} {', '.join(result['categories']):30} {text}")

## 5. Сохранить модель для API

In [ ]:
(ML / "models").mkdir(exist_ok=True)
baseline.save(model, ML / "models" / "baseline.joblib")